<a href="https://colab.research.google.com/github/Krishna-web-hub/PD-lab/blob/main/Copy_of_build_vision_transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Vision Transformer Model

In [ ]:
import torch
import torch.nn as nn # nn sub-function that contains lots of loss fun
import torch.nn.functional as f
import torch.optim as optim # optim contains lots of optimizers functions
from torch.utils.data import dataloader
from torchvision import datasets,transforms
import numpy as np
import torchvision
import random
import matplotlib.pyplot as plt


In [ ]:
torch.__version__ # tocheck version

'2.10.0+cu128'

In [ ]:
torchvision.__version__

'0.25.0+cu128'

In [ ]:
# set device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
!nvidia-smi

Wed Mar 25 10:08:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
torch.cuda.is_available()

True

In [ ]:
print(f" using device:{device}")

 using device:cuda


### set the seed

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

### setting hyperparameter



Increase these

EPOCHS: 10 → 100–200
ViTs on CIFAR-10 typically need 100+ epochs to converge and reach high accuracy.
​

BATCH_SIZE: 128 → 256 (or 512 if GPU allows)
Larger batches give more stable gradients and work well with transformers on CIFAR-10.
​

MLP_DIM: 512 → 1024
Using about 4× EMBED_DIM in the MLP is standard and improves capacity.

​EMBED_DIM: 256 → 384

NUM_HEADS: 8 → 12

Optionally EMBED_DIM & NUM_HEADS:



EMBED_DIM: 256 → 384

NUM_HEADS: 8 → 12
This keeps per-head dim reasonable and can improve attention expressiveness.
​

Keep (do not change for now)

PATCH_SIZE = 4, IMAGE_SIZE = 32, NUM_CLASSES = 10, CHANNELS = 3
Patch size 4 on 32×32 is a good choice for CIFAR-10.
​

LEARNING_RATE = 3e-4 (but add warmup + decay schedule rather than changing the value).
​

DROP_RATE = 0.1 and DEPTH = 6 are reasonable for a small ViT on CIFAR-10

In [ ]:
BATCH_SIZE = 256 # play with  chanaging parameters
EPOCHS = 100
LEARNING_RATE = 3e-4
PATCH_SIZE=4
NUM_CLASSES=10
IMAGE_SIZE= 32
CHANNELS= 3
EMBED_DIM = 384
NUM_HEADS = 12
DEPTH = 4
MLP_DIM= 512
DROP_RATE= 0.3

###  5.DEFINE IMAGE TRANDFORMATIONS

In [ ]:
# 5.DEFINE IMAGE TRANDFORMATIONS commenting below line for finr tuning
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5)) # normalize is mean and median
    # helps model to converge faster / helps to make the numericalcomputation stable
])

"""# fine tuning try changing the parametes to see results
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),


    transforms.ColorJitter(brightness = 0.2 ,
                           contrast= 0.2,
                           saturation=0.2,
                           hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
                           ])   """

'# fine tuning try changing the parametes to see results\ntransform_train = transforms.Compose([\n    transforms.RandomCrop(32, padding=4),\n    transforms.RandomHorizontalFlip(),\n    transforms.RandomVerticalFlip(),\n\n\n    transforms.ColorJitter(brightness = 0.2 ,\n                           contrast= 0.2,\n                           saturation=0.2,\n                           hue=0.2),\n    transforms.ToTensor(),\n    transforms.Normalize((0.5), (0.5))\n                           ])   '

In [ ]:
1e-4

0.0001

In [ ]:
#6. loading datsets
train_dataset = datasets.CIFAR10(root="data",
                                 train=True,
                                 download=True,
                                 transform=transform)

100%|██████████| 170M/170M [00:13<00:00, 12.6MB/s]


In [ ]:
test_dataset = datasets.CIFAR10(root="data",
                                 train=False,
                                 download=True,
                                 transform=transform)


In [ ]:
train_dataset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [ ]:
test_dataset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [ ]:
len(train_dataset)

50000

In [ ]:
len(test_dataset)

10000

### converting data in to dataloaders


data is in form of pytorch

datasets - convert it in to batches(or mini batches) using Dataloader

reason:

1. more computationally efficient(batch_size of 128)

2. gives more chances to neural network to update its gradients per epoch

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)

test_loader= DataLoader(dataset=test_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=False)

In [ ]:
# let's check what we have got created
print(f"DataLoader : {train_loader,test_loader}")
print(f"Length of train_loader:{len(train_loader)} batches of{BATCH_SIZE}...")
print(f"Length of test_loader:{len(test_loader)} batches of {BATCH_SIZE}...")

DataLoader : (<torch.utils.data.dataloader.DataLoader object at 0x78a96849cb00>, <torch.utils.data.dataloader.DataLoader object at 0x78a968264a70>)
Length of train_loader:196 batches of256...
Length of test_loader:40 batches of 256...


### Building VIT transformer model

In [ ]:
class patchEmbedding(nn.Module):
  def __init__(self,
              img_size,
              patch_size,
              in_channels,
              embed_dim):
    super().__init__()
    self.patch_size= patch_size
    self.proj = nn.Conv2d(in_channels=in_channels,
                          out_channels=embed_dim,
                          kernel_size=patch_size,
                          stride=patch_size)
    num_patches = (img_size//patch_size)**2
    self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))
    self.pos_embed = nn.Parameter(torch.randn(1,1+num_patches,embed_dim))

  def forward(self ,x:torch.Tensor):
    B = x.size(0)
    x = self.proj(x)
    x= x.flatten(2).transpose(1,2)
    cls_token = self.cls_token.expand(B,-1,-1) # Corrected: expand cls_token to match batch size
    x = torch.cat((cls_token,x),dim = 1 )
    x = x + self.pos_embed
    return x

In [ ]:
PATCH_SIZE

4

In [ ]:
class MLP(nn.Module):
  def __init__(self,
              in_features,
              hidden_features,
              drop_rate):
    super().__init__()
    self.fc1 = nn.Linear(in_features = in_features,
                         out_features = hidden_features)

    self.fc2 = nn.Linear(in_features= hidden_features,
                         out_features=in_features)
    self.dropout = nn.Dropout(drop_rate)


  def forward (self,x):
    x= self.dropout(f.gelu(self.fc1(x)))
    x = self.dropout(self.fc2(x))
    return x



In [ ]:
class transformerEncoderLayer(nn.Module):
  def __init__(self,
              embed_dim,
              num_heads,
              mlp_dim,
              drop_rate):
    super().__init__()
    self.norm1 = nn.LayerNorm(embed_dim)
    self.attn = nn.MultiheadAttention(embed_dim,
                                      num_heads,
                                       dropout=drop_rate,
                                       batch_first= True)

    self.norm2 = nn.LayerNorm(embed_dim)
    self.mlp = MLP(embed_dim,mlp_dim,drop_rate)

  def forward(self,x):
    x= x + self.attn(self.norm1(x), self.norm1(x),
                     self.norm1(x))[0]
    x = x + self.mlp(self.norm2(x))
    return x




In [ ]:
class VisionTransformer(nn.Module):
  def __init__(self, # these are input that we need tomake vision transformer
               img_size,
               patch_size,
               in_channels,
               num_classes,
               embed_dim,
               depth,
               num_heads,
               mlp_dim,
               drop_rate):
    super().__init__()
    self.patch_embed = patchEmbedding(img_size,
                                      patch_size,
                                      in_channels,
                                      embed_dim)
    self.encoder = nn.Sequential(*[
        transformerEncoderLayer(embed_dim,
                                num_heads,
                                mlp_dim,
                                drop_rate)
        for _ in range(depth)
    ])
    # if we specify depth as 10 that means 10 transformer layers
    self.norm = nn.LayerNorm(embed_dim)
    self.head = nn.Linear(embed_dim,num_classes)

  def forward(self,x):
    x = self.patch_embed(x)
    x = self.encoder(x)
    x = self.norm(x)
    cls_token = x[:,0]
    return self.head(cls_token)

In [ ]:
model.state_dict() # we will get trained weights

NameError: name 'model' is not defined

In [ ]:
# instantiate model
model = VisionTransformer(IMAGE_SIZE,
                          PATCH_SIZE,
                          CHANNELS,
                          NUM_CLASSES,
                          EMBED_DIM,
                          DEPTH,
                          NUM_HEADS,
                          MLP_DIM,
                          DROP_RATE
).to(device)

In [ ]:
model # visualization of model

VisionTransformer(
  (patch_embed): patchEmbedding(
    (proj): Conv2d(3, 384, kernel_size=(4, 4), stride=(4, 4))
  )
  (encoder): Sequential(
    (0): transformerEncoderLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=384, out_features=512, bias=True)
        (fc2): Linear(in_features=512, out_features=384, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
      )
    )
    (1): transformerEncoderLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
      )
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP

### define loss function and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss() # how wrong our model is
optimizer = torch.optim.Adam(model.parameters(), # willupdate model parameter to reduce loss
                             lr=LEARNING_RATE)

In [ ]:
criterion

CrossEntropyLoss()

In [ ]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0
)

In [ ]:
def train(model,loader, optimizer,criterion):
  # set the mode of the model in to training
  model.train()

  total_loss , correct = 0,0

  for x, y in loader:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()##


    out = model(x) # forward pass(model output raw logits)
    loss = criterion(out,y) # calculate loss

    loss.backward() # perform backpropagation

    optimizer.step() # update weights gradient descent

    total_loss += loss.item() * x.size(0) #

# argmax return indesx of highest probablity
    correct += (out.argmax(1) == y).sum().item()

  return total_loss / len(loader.dataset), correct / len(loader.dataset)# normalization the loss general across all batches

In [ ]:
# evaluate model

def evaluate(model,loader,criterion):
  model.eval() # set the mode of the model in to evalauation
  correct = 0
  with torch.inference_mode():
    for x ,y  in loader:
      x , y = x.to(device), y.to(device)
      out = model(x)
      correct += (out.argmax(1) == y).sum().item()
  return correct / len(loader.dataset)


### Training model

In [ ]:
EPOCHS

100

In [ ]:
from tqdm.auto import tqdm

In [ ]:
train_accuracies ,test_accuracies = [],[]

for epoch in  tqdm(range(EPOCHS)):
  train_loss , train_acc = train(model,train_loader,optimizer,criterion)
  test_acc = evaluate(model,test_loader,criterion)
  train_accuracies.append(train_acc)
  test_accuracies.append(test_acc)
  print(f"Epoch:{epoch+1}/{EPOCHS},Train loss:{train_loss:.4f},Train acc:{train_acc:.4f}%,Test acc:{test_acc:.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch:1/100,Train loss:1.7923,Train acc:0.3529%,Test acc:0.4315
Epoch:2/100,Train loss:1.4864,Train acc:0.4657%,Test acc:0.5057
Epoch:3/100,Train loss:1.3655,Train acc:0.5079%,Test acc:0.5271
Epoch:4/100,Train loss:1.2810,Train acc:0.5437%,Test acc:0.5562
Epoch:5/100,Train loss:1.2131,Train acc:0.5672%,Test acc:0.5761
Epoch:6/100,Train loss:1.1539,Train acc:0.5892%,Test acc:0.5792
Epoch:7/100,Train loss:1.0970,Train acc:0.6087%,Test acc:0.5969


In [ ]:
train_accuracies

In [ ]:
test_accuracies

In [ ]:
#plotting
plt.plot(train_accuracies, label="Train accuracy")
plt.plot(test_accuracies, label="Test accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('traininig and test accuracy')
plt.show()

In [ ]:
import random

In [ ]:
test_dataset[0]

In [ ]:
test_dataset[0][0].shape

In [ ]:
test_dataset[0][0].unsqueeze(dim=0).shape # for batch size =1

In [ ]:
test_dataset[0][0] / 2 + 0.5 # converting -ve value to =ve B/C matplotlib  deals with =ve value only

In [ ]:
def predict_and_plot_grid(model,
                          dataset,
                          classes,
                          grid_size=3):
  model.eval()
  fig , axes = plt.subplots(grid_size,grid_size,figsize=(10,10))
  for i in range(grid_size):
    for j in range(grid_size):
      idx = random.randint(0,len(dataset) -1) # -1 for excluding last
      img , true_label = dataset[idx]
      input_tensor = img.unsqueeze(dim=0).to(device)
      with torch.inference_mode(): # infernece to turn  off back propagations
           output = model(input_tensor)
           _, predicted = torch.max(output.data,1)
      img = img / 2 +0.5 # un-normalized
      npimg = img.cpu().numpy() # B/C numpy always use cpu
      axes[i,j].imshow(np.transpose(npimg,(1,2,0)))
      truth = classes[true_label] == classes[predicted.item()]
      if truth:
        color = 'g'
      else:
        color = 'r'

      axes[i,j].set_title(f"Truth: {classes[true_label]}\n,predicted:{classes[predicted.item()]}",fontsize=10,c=color)
      axes[i,j].axis('off')
  plt.tight_layout()
  plt.show()

In [ ]:
predict_and_plot_grid(model,test_dataset,
                      classes=train_dataset.classes,
                      grid_size=3)

In [ ]:
predict_and_plot_grid(model,test_dataset,
                      classes=train_dataset.classes,
                      grid_size=4)

### Augumentataion to get better results


using torchvision transforms using pytorch site reading materils